In [1]:
import warnings

# Ignore FutureWarnings and warnings related to inplace operations
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*inplace.*")

import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from data_collection import collect_stock_data, tickers_earnings_long, tickers_historical_sd

Calculates the number of trading days required for multiple stock tickers'
price to settle within its firm-specific historical volatility band for N
consecutive trading days following each quarterly earnings announcement, for each year.

**Algorithm:**
1. Beginning the day after each earnings announcement, track
   daily price returns.
2. A price is considered "stable" when its absolute daily
   return falls within ± one firm-specific historical standard
   deviation (σ) for N consecutive trading days.
3. Settlement time = total days elapsed − N confirmation days

**Key Parameters:**
- tickers: array of tickers
- year: integer value of the year of investigation
- N: minimum consecutive days within band required to confirm
  equilibrium (e.g. N = 3)

**Output:** Single integer per earnings event representing
the number of trading days to price discovery completion

In [2]:
def price_discovery_completion_metric(
        tickers: list[str],
        year: int,
        interval:str,
        N: int
    ) -> pd.DataFrame:
    """
    Calculates the number of trading days required for each ticker's
    price to settle within its firm-specific historical volatility
    band for N consecutive trading days following each quarterly
    earnings announcement.

    Returns:
        pd.DataFrame with columns: Ticker, Quarter, Earnings Date,
        Settlement Days
    """
    tickers_data = collect_stock_data(tickers, year, interval)
    earnings_dates = tickers_earnings_long(tickers, year)
    historical_sd_df = tickers_historical_sd(tickers, earnings_dates, interval)

    results = []
    for ticker in tickers:
        ticker_data = tickers_data[tickers_data.index.get_level_values('Ticker') == ticker]
        historical_sd_per_ticker = historical_sd_df[historical_sd_df['Ticker'] == ticker]

        for quarter in range(1, 5):
            historical_sd_row = historical_sd_per_ticker[
                historical_sd_per_ticker['Quarter'] == quarter
            ]

            if historical_sd_row.empty:
                print(f"* No historical SD for {ticker} Q{quarter}")
                continue
            
            # Accesses historical_sd for stock
            historical_sd_series = historical_sd_row['Historical SD']
            historical_sd = historical_sd_series.iloc[0]

            announcement_date = historical_sd_row['Earnings Date'].iloc[0]
            # Accounts for after market quarterly earnings announcements, after 4 PM EST
            cutoff = announcement_date.hour if announcement_date < pd.Timestamp(16) else announcement_date + pd.Timedelta(days=1)

            # Get all trading days on/after the announcement, up to a max window
            post_announcement = ticker_data[
                ticker_data.index.get_level_values('Date') >= cutoff
            ].sort_index()

            if post_announcement.empty:
                print(f"* No post-announcement data for {ticker} Q{quarter}")
                continue

            # Daily returns after announcement
            daily_returns = post_announcement['Close'].pct_change().abs()
            daily_returns.name = 'Return'

            consecutive_count = 0
            settlement_day = None
            trigger_event = False
            upper_price = None
            lower_price = None

            for i, ret in enumerate(daily_returns):
                if pd.isna(ret):
                    continue

                if trigger_event:
                    high = post_announcement['High'].iloc[i]
                    low = post_announcement['Low'].iloc[i]

                    within_bounds = (low > lower_price) and (high < upper_price)

                    if within_bounds:
                        consecutive_count += 1
                        if consecutive_count >= N:
                            settlement_day = i - N + 1
                            break
                    else:
                        trigger_event = False
                        consecutive_count = 0

                elif ret <= historical_sd:
                    trigger_event = True
                    trigger_price = post_announcement['Close'].iloc[i]
                    upper_price = trigger_price * (1 + historical_sd)
                    lower_price = trigger_price * (1 - historical_sd)

            if settlement_day is None:
                print(f"Trigger event was never found for {ticker} Q{quarter}")

            results.append({
                'Ticker': ticker,
                'Quarter': quarter,
                'Earnings Date': announcement_date,
                'Settlement Days': settlement_day
            })

    return pd.DataFrame(results)

Testing **price_discovery_completion_metric()** Function

In [3]:
tickers = ['META', 'NVDA', 'AAPL', 'MSFT']
year = 2013
N = 2
interval = '1d'
df = price_discovery_completion_metric(tickers, year, interval, N)
print(df)

TypeError: Cannot compare tz-naive and tz-aware timestamps

Loop through each year (key) and it's tickers (values) to get a DataFrame of each year's tickers' days to settlement (price disovery completion metric)

**Algorithm:**
1. Accesses dictionary values--- year (key) and tickers (value)--- to collect necessary tickers' data (e.g. earnings)
2. Runs through the price_discovery_completion metric() function, creating a table for each year's TICKERS
3. Appends the each yearly table's results into one DataFrame

**Key Parameters:**
1. yearly_tickers: dictionary of stock values with their respective years
2. interval: time frame the stock OHLCV is taken (e.g. '1h', "1d", '5d', '1wk', '1mo')
3. N: number of consecutive days until settlement completion is met

**Output:** Dataframe that includes every year's stock time of settlement given the N consecutive days parameter (row: stock ticker; column: Quarter, Earnings Date, Time Till Settlement Days)

In [ ]:
def yearly_ticker_evaluation(
        yearly_ticker: dict[str],
        interval: str,
        N: int
        ) -> pd.DataFrame:
    all_result = []

    for year, tickers in yearly_ticker.items():
        year_int = int(year)
        yearly_price_metric = price_discovery_completion_metric(tickers, year_int, interval, N)

        if yearly_price_metric is None or yearly_price_metric.empty:
            print(f"* No results for year {year_int}")
            continue

        yearly_price_metric = yearly_price_metric.copy()
        yearly_price_metric['Year'] = year_int
        all_result.append(yearly_price_metric)

    if not all_result:
        return pd.DataFrame()

    return pd.concat(all_result, ignore_index=True)

Testing **yearly_ticker_evaluation()** Function

In [ ]:
yearly_tickers = {
    '2020': ['META', 'AAPL'],
    '2019': ['META', 'AAPL'],
    '2010': ['NVDA', 'WMT']
    }
interval = '1d'
N = 1
df = yearly_ticker_evaluation(yearly_tickers, interval, N)
print(df)

Plots **Boxplot** for each stock's yearly settlement data.

In [ ]:
def boxplot_yearly_settlement_period(
    df: pd.DataFrame
    ) -> None:
    """
    df: Dataframe that contains the settlement period data for each yearly tickers
    """

    df.boxplot(column='Settlement Days', by='Year', grid=False)
    plt.title('Settlement Periods by Year')
    plt.suptitle('')
    plt.xlabel('Year')
    plt.ylabel('Settlement Days')
    plt.show()

Testing **boxplot_yearly_settlement_period()** Function

In [ ]:
# Example scenario
yearly_tickers = {
    '2010': ['KO', 'INTC', 'CAT'],
    '2011': ['JNJ', 'ORCL', 'HD'],
    '2012': ['AAPL', 'PFE', 'MCD'],
    '2013': ['XOM', 'NKE', 'CSCO'],
    '2014': ['MSFT', 'WMT', 'BA'],
    '2015': ['PG', 'AMZN', 'MMM'],
    '2016': ['JPM', 'DIS', 'IBM'],
    '2017': ['NVDA', 'CVX', 'UNH'],
    '2018': ['T', 'MRK', 'GOOGL'],
    '2019': ['VZ', 'AMD', 'GE'],
    '2020': ['COST', 'LMT', 'ADBE'],
}
interval = '1d'
N = 1
df = yearly_ticker_evaluation(yearly_tickers, interval, N)
boxplot_yearly_settlement_period(df)

Plots **Histogram Frequency Graph** ranging from Min and Max Settlement Days

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_histogram_graph(
    df: pd.DataFrame,
    start: int,
    end: int
    ) -> None:
    """
    df: DataFrame with a 'Settlement Days' column (one row per earnings event)
    start: First year, used in the title
    end: Last year, used in the title

    Output: Bar graph of how often each settlement period occurred.
    """
    # Drops empty data points
    days = df['Settlement Days'].dropna().astype(int)
    if days.empty:
        print("No settlement data to plot.")
        return

    counts = days.value_counts().reindex(range(0, days.max() + 1), fill_value=0)

    fig_width = max(8, len(counts) * 0.5)   # 0.5 inch per day, minimum 8
    fig, ax = plt.subplots(figsize=(fig_width, 6))

    ax.bar(counts.index, counts.values, width=0.8, edgecolor='black')

    ax.set_title(f'Frequency of Settlement Periods ({start}-{end})')
    ax.set_xlabel('Settlement Days')
    ax.set_ylabel('Frequency')
    ax.set_xticks(counts.index)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)   # gridlines behind the bars
    plt.xlim(-0.5, len(counts) - 0.5)  # Adjust x-axis limits to fit bars

    plt.show()

    median = days.median()
    mean = days.mean()
    print(f"Median Settlement Days: {median}")
    print(f"Mean Settlement Days: {mean}")

In [ ]:
# Example scenario
yearly_tickers = {
    '2010': ['KO', 'INTC', 'CAT'],
    '2011': ['JNJ', 'ORCL', 'HD'],
    '2012': ['AAPL', 'PFE', 'MCD'],
    '2013': ['XOM', 'NKE', 'CSCO'],
    '2014': ['MSFT', 'WMT', 'BA'],
    '2015': ['PG', 'AMZN', 'MMM'],
    '2016': ['JPM', 'DIS', 'IBM'],
    '2017': ['NVDA', 'CVX', 'UNH'],
    '2018': ['T', 'MRK', 'GOOGL'],
    '2019': ['VZ', 'AMD', 'GE'],
    '2020': ['COST', 'LMT', 'ADBE'],
}
interval = '1d'
N = 1
df = yearly_ticker_evaluation(yearly_tickers, interval, N)

year = list(yearly_tickers.keys())
start = year[0]
end = year[-1]

plot_histogram_graph(df, start, end)